In [1]:
from diffusers import UNet2DModel
import torch



device = "cuda" if torch.cuda.is_available() else "cpu"


model = UNet2DModel(
    sample_size=64,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(128, 256, 512, 512),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),
).to(device)



/export/livia/home/vision/Fbassignana/miniconda3/envs/diffusers-dev/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
x = torch.randn(2, 1, 64, 64, device=device)# [B,C,H,W]
t = torch.randint(0, 1000, (2,), device=device) # [B]

y = model(x, t).sample
print("out:", y.shape)  # (2, 3, 64, 64)

out: torch.Size([2, 1, 64, 64])


In [16]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class NPYImageDataset(Dataset):
    """
    Dataset for loading 1-channel images stored as .npy files.

    Each file should contain:
      - shape (H, W) or (1, H, W)

    Output:
      x: torch.Tensor of shape (1, H, W), dtype=float32
    """

    def __init__(self, root_dir: str, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.files = sorted(
            f for f in os.listdir(root_dir) if f.endswith(".npy")
        )

        if len(self.files) == 0:
            raise RuntimeError(f"No .npy files found in {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.files[idx])
        x = np.load(path)

        # ensure shape (1, H, W)
        if x.ndim == 2:
            x = x[None, ...]
        elif x.ndim == 3 and x.shape[0] == 1:
            pass
        else:
            raise ValueError(f"Invalid shape {x.shape} in {path}")

        x = torch.from_numpy(x).float()
        print(x.shape)
        if self.transform is not None:
            x = self.transform(x)

        return x

In [17]:
   
import torch.nn.functional as F

def transform_256(x: torch.Tensor) -> torch.Tensor:
    return F.interpolate(
        x.unsqueeze(0),          # (1, C, H, W)
        size=(256, 256),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0) 

trainig_set = NPYImageDataset(
    root_dir="../finetuning_sd15/v18/train/",
    transform=transform_256,
)
dataloader = DataLoader(trainig_set, batch_size=2, num_workers=1)

In [20]:
from generative.networks.nets import AutoencoderKL
import torch

vae = AutoencoderKL(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    num_channels=(32, 64, 128),
    num_res_blocks=2,
    attention_levels=(False, False, False),  # ✅ MUST match length
    latent_channels=4,
)



In [22]:
from tqdm import tqdm
pbar = tqdm(total=len(dataloader))

for (idx, batch) in enumerate(dataloader):
    print(batch.shape)
    recon, mu, logvar = vae(batch)
    print(mu.shape, logvar.shape, recon.shape)
    pbar.update(1)

    break

  0%|          | 1/3363 [00:21<19:58:24, 21.39s/it]


torch.Size([1, 192, 256])
torch.Size([1, 192, 256])
torch.Size([1, 192, 256])
torch.Size([1, 192, 256])
torch.Size([1, 192, 256])
torch.Size([1, 192, 256])
torch.Size([2, 1, 256, 256])


  0%|          | 1/3363 [00:00<48:39,  1.15it/s]

torch.Size([2, 4, 64, 64]) torch.Size([2, 4, 64, 64]) torch.Size([2, 1, 256, 256])
